## Cleaning and Formatting Main Content Data 

This notebook cleans and formats the data used to iterate over countries, display overall index and other information about the countries that is not used directly for visualizations or graphs. 

In [6]:
# Import libraries
import json
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

### 1. Add income classification

The income level data is taken from the World Bank classification computed based on GNI values. 




In [7]:
# Read classification dataset
world_bank_df = pd.read_excel("world_bank_income_class.xlsx")

# Drop columns that are not needed and rename the rest to match the main content columns
world_bank_df = world_bank_df.drop(
    columns=["Code", "Region", "Lending category"]
).rename(columns={"Economy": "Name", "Income group": "Income_Class"})

# Remove leading and trailing whitespaces from country name column
world_bank_df["Name"] = world_bank_df["Name"].str.strip()
world_bank_df.head()

,Name,Income_Class
0,Afghanistan,Low income
1,Albania,Upper middle income
2,Algeria,Upper middle income
3,American Samoa,High income
4,Andorra,High income


In [8]:
# Read main context datatset

# Move up two levels to get parent directory
BASE_DIR = Path.cwd()
PARENT_DIR = BASE_DIR.parent.parent

# Get file path of the main content file
file_path = PARENT_DIR / "web-dev" / "app" / "data" / "main-content.xlsx"

# Read dataframe and strip country name column
main_df = pd.read_excel(file_path)
main_df["Name"] = main_df["Name"].str.strip()
main_df.head()

,Name,Prefix,Slug
0,Aland Islands,AX,aland-islands
1,Albania,AL,albania
2,Andorra,AD,andorra
3,Armenia,AM,armenia
4,Austria,AT,austria


In [9]:
# Identify missing values or country names that are written differently
missing_in_world_bank = main_df[~main_df["Name"].isin(world_bank_df["Name"])]
display(missing_in_world_bank)


# Map names and apply the mapping to standardize country names
name_map = {
    "Bosnia and Herzegowina": "Bosnia and Herzegovina",
    "Iran, Islamic Rep.": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Slovak Republic": "Slovakia",
    "West Bank and Gaza": "Palestine",
    "Åland Islands": "Aland Islands",
    "Vatican City State": "Holy See",
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
    "Yemen, Rep": "Yemen",
}

main_df["Name"] = main_df["Name"].replace(name_map)
world_bank_df["Name"] = world_bank_df["Name"].replace(name_map)

,Name,Prefix,Slug
0,Aland Islands,AX,aland-islands
24,Guernsey,GG,guernsey
25,Holy See,VA,holy-see
28,Iran,IR,iran
34,Jersey,JE,jersey
39,Kyrgyzstan,KG,kyrgyzstan
53,Palestine,PS,palestine
62,Slovakia,SK,slovakia
65,Svalbard And Jan Mayen Islands,SJ,svalbard-and-jan-mayen-islands
70,Turkey,TR,turkiye


In [10]:
# Merge datasets
main_df = main_df.merge(world_bank_df, on="Name", how="left")
display(main_df)

main_df.to_excel(file_path, index=False)

,Name,Prefix,Slug,Income_Class
0,Aland Islands,AX,aland-islands,NaN
1,Albania,AL,albania,Upper middle income
2,Andorra,AD,andorra,High income
3,Armenia,AM,armenia,Upper middle income
4,Austria,AT,austria,High income
...,...,...,...,...
72,Ukraine,UA,ukraine,Upper middle income
73,United Arab Emirates,AE,united-arab-emirates,High income
74,United Kingdom,GB,united-kingdom,High income
75,Uzbekistan,UZ,uzbekistan,Lower middle income


### 2. Calculate Internet Infrastructure and Economic Development (IIED) index


#### 2.1 Read and format JSON file values

The JSON file with the unscaled values is used to calculate indices.

In [ ]:
# Load json file with all of the data containing the real, unscaled values
file_path = "real_data.json"  # This file can also be found as website_data.json in the main folder.
with open(file_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df = pd.DataFrame(raw_data)
df = df.rename(columns={"year": "Year"})

df.head()

,Country Name,Year,Economic Index,v4_prefixes_ris,v6_prefixes_ris,asns_ris,v4_prefixes_stats,v6_prefixes_stats,asns_stats,speed (mgps),bandwidth (kbit/s),GDP per Capita,GDP,GNI,GNI per Capita,Foreign Investment,GINI Index,Unemployment Rate,Labor Force participation rate
0,Qatar,2022,0.402156,123555.0,96985.5,4868.0,-348.0,-348.0,6323.0,86.88,255102.7,88701.463350,2.357093e+11,2.274817e+11,71070.0,7.609890e+07,NaN,0.130,87.575
1,Qatar,2023,0.385429,129173.0,118954.0,4515.0,-339.0,-339.0,6658.0,153.89,NaN,80195.874647,2.130028e+11,2.069289e+11,79430.0,-4.741758e+08,NaN,0.130,87.550
2,Luxembourg,2021,0.371657,351576.5,54919.0,27952.0,-351.0,-351.0,40907.0,173.16,NaN,133711.794436,8.558411e+10,5.863045e+10,87610.0,5.217423e+10,32.7,5.571,61.482
3,Qatar,2018,0.367351,105487.0,4389.0,3245.0,-354.0,-354.0,5306.0,32.79,106632.7,71039.849058,1.833350e+11,1.795896e+11,63410.0,-2.186264e+09,NaN,0.110,87.503
4,Qatar,2021,0.364576,115112.0,71760.5,4514.0,-347.0,-347.0,5995.0,109.57,247955.6,71751.883123,1.797320e+11,1.769691e+11,66970.0,-1.093407e+09,NaN,0.140,86.363


In [12]:
# Sort by 'Country Name' and 'Year'
df = df.sort_values(by=["Country Name", "Year"], ascending=[True, True]).reset_index(
    drop=True
)

# Drop columns not needed
columns = [
    "v4_prefixes_stats",
    "v6_prefixes_stats",
    "asns_stats",
    "GDP per Capita",
    "GDP",
    "GNI",
    "GINI Index",
    "Economic Index",
]
df = df.drop(columns, axis=1)
df.head()

,Country Name,Year,v4_prefixes_ris,v6_prefixes_ris,asns_ris,speed (mgps),bandwidth (kbit/s),GNI per Capita,Foreign Investment,Unemployment Rate,Labor Force participation rate
0,Albania,2000,NaN,NaN,NaN,NaN,NaN,1100.0,1.430000e+08,19.023,60.435
1,Albania,2001,NaN,NaN,NaN,NaN,NaN,1280.0,2.073000e+08,18.570,59.888
2,Albania,2002,NaN,NaN,NaN,NaN,NaN,1370.0,1.350000e+08,17.891,59.610
3,Albania,2003,NaN,NaN,NaN,NaN,NaN,1650.0,1.780364e+08,16.985,58.557
4,Albania,2004,NaN,NaN,NaN,NaN,NaN,2100.0,3.412851e+08,16.306,57.496


In [13]:
# Map names and apply the mapping to standardize country names
name_map = {
    "Bosnia and Herzegowina": "Bosnia and Herzegovina",
    "Bosnia And Herzegovina": "Bosnia and Herzegovina",
    "Iran, Islamic Rep.": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Slovak Republic": "Slovakia",
    "West Bank and Gaza": "Palestine",
    "Åland Islands": "Aland Islands",
    "Vatican City State": "Holy See",
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
    "Yemen, Rep.": "Yemen",
}

df["Country Name"] = df["Country Name"].str.strip()
df["Country Name"] = df["Country Name"].replace(name_map)

In [14]:
# Check for duplicates
duplicates = df[df.duplicated(subset=["Country Name", "Year"], keep=False)]
display(duplicates)

# Fill NaN with existing values and drop duplicates
df = df.groupby(["Country Name", "Year"], as_index=False).first()

,Country Name,Year,v4_prefixes_ris,v6_prefixes_ris,asns_ris,speed (mgps),bandwidth (kbit/s),GNI per Capita,Foreign Investment,Unemployment Rate,Labor Force participation rate
1665,Yemen,2015,13138.0,297.0,515.0,NaN,5116.4,NaN,NaN,NaN,NaN
1666,Yemen,2016,22102.0,354.0,708.0,NaN,4883.6,NaN,NaN,NaN,NaN
1667,Yemen,2017,24004.0,831.0,694.0,NaN,4735.4,NaN,NaN,NaN,NaN
1668,Yemen,2018,27008.5,2425.0,924.0,NaN,NaN,NaN,NaN,NaN,NaN
1669,Yemen,2019,30738.0,2880.0,960.0,2.63,NaN,NaN,NaN,NaN,NaN
1670,Yemen,2020,38124.0,2998.0,998.0,4.35,NaN,NaN,NaN,NaN,NaN
1671,Yemen,2021,42638.0,3260.0,1019.0,5.95,NaN,NaN,NaN,NaN,NaN
1672,Yemen,2022,46019.5,2849.0,1013.0,2.72,NaN,NaN,NaN,NaN,NaN
1673,Yemen,2023,46719.0,2863.0,990.0,5.28,NaN,NaN,NaN,NaN,NaN
1689,Yemen,2015,NaN,NaN,NaN,NaN,NaN,1030.0,-1.544481e+07,17.900,32.068


#### 2.2 Integrate Internet Penetration Rate

In [15]:
# Get path of internet penetration rate csv
BASE_DIR = Path.cwd()
PARENT_DIR = BASE_DIR.parent
file_path = BASE_DIR / "internet_penetration_rate.csv"

ipr_df = pd.read_csv(
    file_path,
    skiprows=4,  # Skip non-data rows
    header=0,
)
# Map names and apply the mapping to standardize country names
ipr_df["Country Name"] = ipr_df["Country Name"].replace(name_map)
display(ipr_df.head())

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,Unnamed: 68
0,Aruba,ABW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,88.70,93.5,97.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,8.26,11.0,13.5,16.8,17.6,17.0,16.5,17.2,17.7,NaN
3,Africa Western and Central,AFW,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Individuals using the Internet (% of population),IT.NET.USER.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,22.00,23.2,26.0,29.0,32.1,36.6,39.4,42.1,44.8,NaN


In [16]:
ipr_df = pd.melt(
    ipr_df,
    id_vars=["Country Name"],
    var_name="Year",
    value_name="Internet Penetration Rate",
)

# Convert 'Year' column to numeric values for sorting
ipr_df["Year"] = pd.to_numeric(ipr_df["Year"], errors="coerce")

# Merge dataframes
merged_df = pd.merge(df, ipr_df, on=["Country Name", "Year"], how="left")

display(merged_df.head())

,Country Name,Year,v4_prefixes_ris,v6_prefixes_ris,asns_ris,speed (mgps),bandwidth (kbit/s),GNI per Capita,Foreign Investment,Unemployment Rate,Labor Force participation rate,Internet Penetration Rate
0,Albania,2000,NaN,NaN,NaN,NaN,NaN,1100.0,1.430000e+08,19.023,60.435,0.114
1,Albania,2001,NaN,NaN,NaN,NaN,NaN,1280.0,2.073000e+08,18.570,59.888,0.326
2,Albania,2002,NaN,NaN,NaN,NaN,NaN,1370.0,1.350000e+08,17.891,59.610,0.39
3,Albania,2003,NaN,NaN,NaN,NaN,NaN,1650.0,1.780364e+08,16.985,58.557,0.972
4,Albania,2004,NaN,NaN,NaN,NaN,NaN,2100.0,3.412851e+08,16.306,57.496,2.42


#### 2.3 Calculate Economic Index

Economic Index is developed based on GNI per capita rather than GDP as the metric is used for income level classification.

In [17]:
# Select columns with economic data
columns = [
    "Foreign Investment",
    "GNI per Capita",
    "Unemployment Rate",
    "Labor Force participation rate",
]

# Create dataframe with columns for economic index
econ_index = merged_df[["Country Name", "Year"] + columns].copy()

# Check for missing values
print("Missing values before filling:", econ_index.isna().sum())

# Impute missing data based on historical country data
econ_index[columns] = econ_index.groupby("Country Name")[columns].transform(
    lambda x: x.fillna(x.mean())
)

# Check again for any missing values
print("Missing values after country-level filling:", econ_index.isna().sum())

# Normalize the data (Min-Max Scaling) to standardize values
scaler = MinMaxScaler()
econ_index[columns] = scaler.fit_transform(econ_index[columns])


# Define weights
weights = {
    "Foreign Investment": 0.1,
    "GNI per Capita": 0.5,  # Positive weight (higher GNI is good)
    "Unemployment Rate": -0.2,  # Negative weight (higher unemployment is bad)
    "Labor Force participation rate": 0.2,
}

# Compute Economic Index
econ_index["GNI_Economic_Index"] = sum(
    econ_index[col] * weight for col, weight in weights.items()
)

Missing values before filling: Country Name                        0
Year                                0
Foreign Investment                227
GNI per Capita                    134
Unemployment Rate                 180
Labor Force participation rate    180
dtype: int64
Missing values after country-level filling: Country Name                        0
Year                                0
Foreign Investment                177
GNI per Capita                     57
Unemployment Rate                 177
Labor Force participation rate    177
dtype: int64


In [18]:
# Show top countries based on index
top_values = econ_index.sort_values(by="GNI_Economic_Index", ascending=False).head(10)
display(top_values)

,Country Name,Year,Foreign Investment,GNI per Capita,Unemployment Rate,Labor Force participation rate,GNI_Economic_Index
1222,Qatar,2013,0.295509,0.793888,0.004836,0.976958,0.620919
1223,Qatar,2014,0.297313,0.758071,0.002687,0.985083,0.605246
1221,Qatar,2012,0.296695,0.762014,0.010210,0.976489,0.603933
1220,Qatar,2011,0.297215,0.687834,0.012359,0.979822,0.567131
1102,Norway,2013,0.295003,0.859854,0.089334,0.610157,0.563592
1103,Norway,2014,0.299424,0.861825,0.090973,0.602609,0.563182
1224,Qatar,2015,0.297342,0.658917,0.001881,0.994577,0.557732
1232,Qatar,2023,0.295860,0.651195,0.000806,0.999550,0.554932
1112,Norway,2023,0.306596,0.844081,0.093283,0.597096,0.553463
1101,Norway,2012,0.323067,0.818122,0.081381,0.612842,0.547660


#### 2.4 Calculate Internet Index

In [19]:
pd.set_option("future.no_silent_downcasting", True)

# Select columns with internet data
columns = [
    "v4_prefixes_ris",
    "v6_prefixes_ris",
    "asns_ris",
    "speed (mgps)",
    "bandwidth (kbit/s)",
    "Internet Penetration Rate",
]

# Create dataframe for internet index
internet_index = merged_df[["Country Name", "Year"] + columns].copy()

# Check for missing values
print("Missing values before filling:", internet_index.isna().sum())

# Impute missing data based on historical country data
internet_index[columns] = internet_index.groupby("Country Name")[columns].transform(
    lambda x: x.fillna(x.mean())
)

# Check again for any missing values
print("Missing values after country-level filling:", internet_index.isna().sum())

# Normalize the data (Min-Max Scaling) to get a standardized values
scaler = MinMaxScaler()
internet_index[columns] = scaler.fit_transform(internet_index[columns])


# Define weights
weights = {
    "v4_prefixes_ris": 0.16,
    "v6_prefixes_ris": 0.16,
    "asns_ris": 0.16,
    "speed (mgps)": 0.16,
    "bandwidth (kbit/s)": 0.16,
    "Internet Penetration Rate": 0.16,
}

# Compute internet index
internet_index["Internet_Index"] = sum(
    internet_index[col] * weight for col, weight in weights.items()
)

Missing values before filling: Country Name                    0
Year                            0
v4_prefixes_ris              1095
v6_prefixes_ris              1095
asns_ris                     1095
speed (mgps)                 1268
bandwidth (kbit/s)           1348
Internet Penetration Rate      88
dtype: int64
Missing values after country-level filling: Country Name                   0
Year                           0
v4_prefixes_ris              120
v6_prefixes_ris              120
asns_ris                     120
speed (mgps)                 177
bandwidth (kbit/s)           177
Internet Penetration Rate     24
dtype: int64


In [20]:
# Check top values
top_internet = internet_index.sort_values(by="Internet_Index", ascending=False).head(10)
display(top_internet)

,Country Name,Year,v4_prefixes_ris,v6_prefixes_ris,asns_ris,speed (mgps),bandwidth (kbit/s),Internet Penetration Rate,Internet_Index
1280,Russian Federation,2023,1.000000,0.577360,0.945705,0.325622,0.006465,0.921962,0.604338
1278,Russian Federation,2021,0.963694,0.436756,0.970714,0.378992,0.006465,0.881943,0.582170
1279,Russian Federation,2022,0.996418,0.463234,0.957763,0.289654,0.006465,0.903953,0.578798
1277,Russian Federation,2020,0.962388,0.420985,1.000000,0.306475,0.006465,0.849927,0.567398
1276,Russian Federation,2019,0.869849,0.354767,0.979259,0.225526,0.007404,0.825915,0.522035
1271,Russian Federation,2014,0.847978,0.352574,0.952199,0.260907,0.006465,0.704857,0.499997
1270,Russian Federation,2013,0.847978,0.352574,0.952199,0.260907,0.006465,0.679844,0.495995
1269,Russian Federation,2012,0.847978,0.352574,0.952199,0.260907,0.006465,0.659835,0.492793
1064,Netherlands,2023,0.302868,1.000000,0.196410,0.557461,0.013647,0.969985,0.486460
462,Germany,2021,0.437140,0.709155,0.425828,0.534286,0.005711,0.913958,0.484172


#### 2.5 Combine Indices

Combine internet and economic index in a final dataframe.

In [21]:
# Merge internet index
merged_df = merged_df.merge(
    internet_index[["Country Name", "Year", "Internet_Index"]],
    on=["Country Name", "Year"],
    how="left",
)

# Merge economic index
merged_df = merged_df.merge(
    econ_index[["Country Name", "Year", "GNI_Economic_Index"]],
    on=["Country Name", "Year"],
    how="left",
)

merged_df.head(10)

,Country Name,Year,v4_prefixes_ris,v6_prefixes_ris,asns_ris,speed (mgps),bandwidth (kbit/s),GNI per Capita,Foreign Investment,Unemployment Rate,Labor Force participation rate,Internet Penetration Rate,Internet_Index,GNI_Economic_Index
0,Albania,2000,NaN,NaN,NaN,NaN,NaN,1100.0,1.430000e+08,19.023,60.435,0.114,0.026321,0.034035
1,Albania,2001,NaN,NaN,NaN,NaN,NaN,1280.0,2.073000e+08,18.570,59.888,0.326,0.026661,0.035244
2,Albania,2002,NaN,NaN,NaN,NaN,NaN,1370.0,1.350000e+08,17.891,59.610,0.39,0.026763,0.038253
3,Albania,2003,NaN,NaN,NaN,NaN,NaN,1650.0,1.780364e+08,16.985,58.557,0.972,0.027695,0.040482
4,Albania,2004,NaN,NaN,NaN,NaN,NaN,2100.0,3.412851e+08,16.306,57.496,2.42,0.030013,0.042171
5,Albania,2005,NaN,NaN,NaN,NaN,NaN,2620.0,2.624790e+08,15.966,56.428,6.04,0.035807,0.042278
6,Albania,2006,NaN,NaN,NaN,NaN,NaN,3050.0,3.251383e+08,15.626,55.354,9.61,0.041522,0.042008
7,Albania,2007,NaN,NaN,NaN,NaN,NaN,3460.0,6.522756e+08,15.966,54.275,15.0,0.050150,0.038009
8,Albania,2008,NaN,NaN,NaN,NaN,NaN,4040.0,1.247182e+09,13.060,53.192,23.9,0.064397,0.052161
9,Albania,2009,NaN,NaN,NaN,NaN,NaN,4280.0,1.345415e+09,13.674,54.993,41.2,0.092091,0.056346


#### 2.6 Calculate Overall Index, Ranking and Score Changes

The overall index uses 2015 as the earliest year. While using data from 2000 is beneficial for long-term insights in the development of machine learning models, the data displayed on the website starts from 2015 to show the most relevant trends. 

In [22]:
# Create dataframe with values from indices (2015 and 2023 only)
df = merged_df[["Country Name", "Year", "GNI_Economic_Index", "Internet_Index"]]
df = df.rename(columns={"Country Name": "Name"})
data = df[(df["Year"] == 2023) | (df["Year"] == 2015)]

# See data per country
data = data.pivot(
    index="Name", columns="Year", values=["GNI_Economic_Index", "Internet_Index"]
)
data.columns = [f"{col[0]}_{col[1]}" for col in data.columns]
data.head(20)

,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023
Name,,,,
Albania,0.039671,0.109097,0.113896,0.172947
Andorra,NaN,NaN,0.216450,0.217493
Armenia,0.054903,0.094213,0.119226,0.166627
Austria,0.294274,0.331256,0.205356,0.254289
Azerbaijan,0.143785,0.139994,0.140950,0.168299
Bahrain,0.261956,0.280046,0.183199,0.237641
Belarus,0.141720,0.156371,0.140987,0.192794
Belgium,0.246319,0.304180,0.209723,0.247025
Bosnia and Herzegovina,-0.032486,0.068847,0.105883,0.154122


In [23]:
# List of columns to be multiplied and rounded for scaling
columns_to_scale = [
    "Internet_Index_2023",
    "Internet_Index_2015",
    "GNI_Economic_Index_2023",
    "GNI_Economic_Index_2015",
]

# Reset the index
data = data.reset_index()

# Scale columns
data[columns_to_scale] = (data[columns_to_scale] * 100).round(1)
data.head()

,Name,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023
0,Albania,4.0,10.9,11.4,17.3
1,Andorra,NaN,NaN,21.6,21.7
2,Armenia,5.5,9.4,11.9,16.7
3,Austria,29.4,33.1,20.5,25.4
4,Azerbaijan,14.4,14.0,14.1,16.8


In [24]:
# Calculate the Overall Index for 2015
data["Overall_Index_2015"] = (
    data[["GNI_Economic_Index_2015", "Internet_Index_2015"]].mean(axis=1, skipna=False)
).round()

# Calculate the Overall Index for 2023
data["Overall_Index_2023"] = (
    data[["GNI_Economic_Index_2023", "Internet_Index_2023"]].mean(axis=1, skipna=False)
).round(1)
data.head()

,Name,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023,Overall_Index_2015,Overall_Index_2023
0,Albania,4.0,10.9,11.4,17.3,8.0,14.1
1,Andorra,NaN,NaN,21.6,21.7,NaN,NaN
2,Armenia,5.5,9.4,11.9,16.7,9.0,13.0
3,Austria,29.4,33.1,20.5,25.4,25.0,29.2
4,Azerbaijan,14.4,14.0,14.1,16.8,14.0,15.4


In [25]:
# Calculate the rank of each country based on the highest Overall Index score to the lowest
data["Rank"] = data["Overall_Index_2023"].rank(method="first", ascending=False)
data["Rank"] = data["Rank"].apply(lambda x: int(x) if pd.notna(x) else "Unknown")

# Find score change by calculating differences from the index in 2015 and 2023
data["Score_Change"] = data["Overall_Index_2023"] - data["Overall_Index_2015"]

data = data.fillna("Unknown")
display(data)

,Name,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023,Overall_Index_2015,Overall_Index_2023,Rank,Score_Change
0,Albania,4.0,10.9,11.4,17.3,8.0,14.1,47,6.1
1,Andorra,Unknown,Unknown,21.6,21.7,Unknown,Unknown,Unknown,Unknown
2,Armenia,5.5,9.4,11.9,16.7,9.0,13.0,49,4.0
3,Austria,29.4,33.1,20.5,25.4,25.0,29.2,18,4.2
4,Azerbaijan,14.4,14.0,14.1,16.8,14.0,15.4,45,1.4
...,...,...,...,...,...,...,...,...,...
66,Ukraine,7.9,9.1,21.4,29.5,15.0,19.3,39,4.3
67,United Arab Emirates,38.6,38.5,23.0,35.4,31.0,37.0,9,6.0
68,United Kingdom,29.5,30.3,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
69,Uzbekistan,10.3,10.3,8.7,18.0,10.0,14.2,46,4.2


#### 3. Map Color Coding

Build map color coding based on index values. The map is displayed on the homepage and shows the internet infrastructure and economic development index across all countries.

In [26]:
def map_colors(value):
    if value == "Unknown":
        return "#acaaaa"
    elif value <= 10:
        return "#580000"
    elif 10 < value <= 15:
        return "#ae431b"
    elif 15 < value <= 20:
        return "#ffd6a8"
    elif 20 < value <= 25:
        return "#bbd7c8"
    elif 25 < value <= 30:
        return "#3d7a8c"
    elif value >= 30:
        return "#131f48"


# Apply map colors
data["Color_Overall_2023"] = data["Overall_Index_2023"].apply(map_colors)

display(data)

,Name,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023,Overall_Index_2015,Overall_Index_2023,Rank,Score_Change,Color_Overall_2023
0,Albania,4.0,10.9,11.4,17.3,8.0,14.1,47,6.1,#ae431b
1,Andorra,Unknown,Unknown,21.6,21.7,Unknown,Unknown,Unknown,Unknown,#acaaaa
2,Armenia,5.5,9.4,11.9,16.7,9.0,13.0,49,4.0,#ae431b
3,Austria,29.4,33.1,20.5,25.4,25.0,29.2,18,4.2,#3d7a8c
4,Azerbaijan,14.4,14.0,14.1,16.8,14.0,15.4,45,1.4,#ffd6a8
...,...,...,...,...,...,...,...,...,...,...
66,Ukraine,7.9,9.1,21.4,29.5,15.0,19.3,39,4.3,#ffd6a8
67,United Arab Emirates,38.6,38.5,23.0,35.4,31.0,37.0,9,6.0,#131f48
68,United Kingdom,29.5,30.3,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,#acaaaa
69,Uzbekistan,10.3,10.3,8.7,18.0,10.0,14.2,46,4.2,#ae431b


#### 4. Update main content

Update the file with the main content to include values regarding index and colors for map.

In [27]:
# Load the file with the main content used for website
BASE_DIR = Path.cwd()
PARENT_DIR = BASE_DIR.parent.parent

file_path = PARENT_DIR / "web-dev" / "app" / "data" / "main-content.xlsx"
df = pd.read_excel(file_path)
display(df.head())

,Name,Prefix,Slug,Income_Class
0,Aland Islands,AX,aland-islands,NaN
1,Albania,AL,albania,Upper middle income
2,Andorra,AD,andorra,High income
3,Armenia,AM,armenia,Upper middle income
4,Austria,AT,austria,High income


In [28]:
# Map names and apply the mapping to standardize country names
name_map = {
    "Bosnia and Herzegowina": "Bosnia and Herzegovina",
    "Iran, Islamic Rep.": "Iran",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Slovak Republic": "Slovakia",
    "West Bank And Gaza": "Palestine",
    "Åland Islands": "Aland Islands",
    "Vatican City State": "Holy See",
    "Türkiye": "Turkey",
    "Turkiye": "Turkey",
    "Yemen, Rep": "Yemen",
}

data["Name"] = data["Name"].replace(name_map)
df["Name"] = df["Name"].replace(name_map)

In [29]:
# Merge dataframes with economic data and raw data
merged_df = df.merge(data, on="Name", how="outer")
merged_df = merged_df.fillna("Unknown")
display(merged_df)

,Name,Prefix,Slug,Income_Class,GNI_Economic_Index_2015,GNI_Economic_Index_2023,Internet_Index_2015,Internet_Index_2023,Overall_Index_2015,Overall_Index_2023,Rank,Score_Change,Color_Overall_2023
0,Aland Islands,AX,aland-islands,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
1,Albania,AL,albania,Upper middle income,4.0,10.9,11.4,17.3,8.0,14.1,47,6.1,#ae431b
2,Andorra,AD,andorra,High income,Unknown,Unknown,21.6,21.7,Unknown,Unknown,Unknown,Unknown,#acaaaa
3,Armenia,AM,armenia,Upper middle income,5.5,9.4,11.9,16.7,9.0,13.0,49,4.0,#ae431b
4,Austria,AT,austria,High income,29.4,33.1,20.5,25.4,25.0,29.2,18,4.2,#3d7a8c
...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,Ukraine,UA,ukraine,Upper middle income,7.9,9.1,21.4,29.5,15.0,19.3,39,4.3,#ffd6a8
73,United Arab Emirates,AE,united-arab-emirates,High income,38.6,38.5,23.0,35.4,31.0,37.0,9,6.0,#131f48
74,United Kingdom,GB,united-kingdom,High income,29.5,30.3,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,#acaaaa
75,Uzbekistan,UZ,uzbekistan,Lower middle income,10.3,10.3,8.7,18.0,10.0,14.2,46,4.2,#ae431b


In [30]:
# Move up two levels to get parent directory
PARENT_DIR = BASE_DIR.parent.parent

# Write path of json file and save to add it in the web development folder where it is extracted for visualizations
file_path = PARENT_DIR / "web-dev" / "app" / "data" / "main-content.xlsx"
merged_df = merged_df.reset_index(drop=True)
merged_df.to_excel(file_path, index=False)